### Setup

In [ ]:
"""Google colab requirements"""
!pip install torch_geometric numpy torchmetrics

In [ ]:
import pandas as pd
import os
import json
import numpy as np
from os.path import dirname
pd.set_option("display.max_columns", None)

DATASET         = "bpi_2012"

ROOT_PATH       = dirname(os.getcwd())      
ROOT_PATH       = "drive/MyDrive/Thesis"    # for colab
PROCESSED_PATH  = f"{ROOT_PATH}/data/datasets/processed/{DATASET}"
MODEL_PATH      = f"{ROOT_PATH}/data/datasets/models/{DATASET}"
GRAPHS_PATH     = f"{ROOT_PATH}/data/datasets/graphs/{DATASET}"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# import sys
# sys.path.append(f"{ROOT_PATH}/data")

In [ ]:
with open(f"{ROOT_PATH}/data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

list(datasets_info.keys())

In [ ]:
tab_all = pd.read_csv(f"{PROCESSED_PATH}/{DATASET}_processed_all.csv")
tab_all.head()

In [ ]:
dataset_info = datasets_info[DATASET]
dataset_info

In [ ]:
tab_train = pd.read_csv(f"{PROCESSED_PATH}/{DATASET}_processed_train.csv")
tab_valid = pd.read_csv(f"{PROCESSED_PATH}/{DATASET}_processed_valid.csv")
tab_test = pd.read_csv(f"{PROCESSED_PATH}/{DATASET}_processed_test.csv")

In [ ]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

print(categorical_columns)
print(real_value_columns)

In [80]:
for k in categorical_columns:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")
    
 

In [ ]:
if DATASET == "sp2020":
    tab_all["REPAIR_IN_TIME_5D"] = [float(x) if x is not np.nan else x for x in tab_all["REPAIR_IN_TIME_5D"].values]
    tab_train["REPAIR_IN_TIME_5D"] = [float(x) if x is not np.nan else x for x in tab_train["REPAIR_IN_TIME_5D"].values]
    tab_valid["REPAIR_IN_TIME_5D"] = [float(x) if x is not np.nan else x for x in tab_valid["REPAIR_IN_TIME_5D"].values]
    tab_test["REPAIR_IN_TIME_5D"] = [float(x) if x is not np.nan else x for x in tab_test["REPAIR_IN_TIME_5D"].values]

In [ ]:
from math import log

def log_norm(x):
    return log(x+1)

if DATASET == "BPI20_RequestForPayment_CZ":
    tab_all["case:RequestedAmount"] = tab_all["case:RequestedAmount"].apply(log_norm)
    tab_train["case:RequestedAmount"] = tab_train["case:RequestedAmount"].apply(log_norm)
    tab_valid["case:RequestedAmount"] = tab_valid["case:RequestedAmount"].apply(log_norm)
    tab_test["case:RequestedAmount"] = tab_test["case:RequestedAmount"].apply(log_norm)
   

### Prepare the graphs

In [ ]:
from utils import get_case_ids, get_one_hot_encodings
from torch import tensor,int64, float32
from torch_geometric.data import HeteroData


In [ ]:
import sklearn.preprocessing

""" Create onehot encoder for the column types"""
def get_one_hot_encoder(dataset: pd.DataFrame, key: str):
    datas = dataset[key].unique()                   # Get all the values a key could have
    if key == "Activity":                           # Add the END node
        datas = np.union1d(datas, ["END"])         
    datas = datas.astype(np.str_)                   # Make everything a string
    datas = datas.reshape([len(datas), 1])          # Add a dimension
    onehot = sklearn.preprocessing.OneHotEncoder()  # Learn and return
    onehot.fit(datas)
    return onehot

In [88]:
ONE_HOT_ENCODERS = {k: get_one_hot_encoder(tab_all, k) for k in categorical_columns}
ONE_HOT_ENCODERS

{'org:resource': OneHotEncoder(),
 'Activity': OneHotEncoder(),
 'org:role': OneHotEncoder(),
 'case:Project': OneHotEncoder(),
 'case:Task': OneHotEncoder(),
 'case:OrganizationalEntity': OneHotEncoder(),
 'case:Activity': OneHotEncoder(),
 'case:RfpNumber': OneHotEncoder()}

In [ ]:
""" Normalize the timestamps (relative duration from the start) """
def ladd_new_timestamp(trace: pd.DataFrame):
    times = list(trace["time:timestamp"].copy())
    for i in range(1,len(times)):
        times[i] = times[i] - times[0]
    times[0] = 0.
    trace2 = trace.copy()
    trace2["time:timestamp"] = times 
    return trace2

In [ ]:
""" Get trace columns and encode it into the correct tensors """
def get_node_features(trace: pd.DataFrame, cat_features, real_features) -> dict:
    columns_static = [c for c in trace if len(set(trace[c])) == 1]

    res = {}

    for key in trace:
        values = trace[key].values
        # Categorical feature handling
        if key in cat_features:
            onehot_encoder = ONE_HOT_ENCODERS[key]
            values = values.astype(np.str_)
            if key not in columns_static:
                try:
                    res[key] = tensor(
                        get_one_hot_encodings(onehot_encoder, values),
                        dtype=float32,
                        requires_grad=True
                    )
                except ValueError:
                    print("Error in the encoding")
                    print(key)
                    print(values)
            else:
                res[key] = tensor(
                    get_one_hot_encodings(onehot_encoder, np.array([values[0]])),
                    dtype=float32,
                    requires_grad=True
                )
        # Numerical features handling
        elif key in real_features:
            if key not in columns_static:
                res[key] = tensor(values,  dtype=float32,requires_grad=True)
            else:
                res[key] = tensor([values[0]], dtype=float32,requires_grad=True)
            res[key] = res[key].reshape(res[key].shape[0], 1)
        
    

    return res

In [ ]:
""" Determines the edge type that links different types of nodes """
def compute_edges_indexs(node_features: dict, prefix_len):
    res = {}
    keys = node_features.keys()
    indexes = [[i, i + 1] for i in range(prefix_len-1)]
    # activities indexes
    for k in keys:
        if len(node_features[k]) != 1:      # Only dynamic features
            if k == "Activity":
                res[(k, "followed_by", k)] = indexes
                for k2 in keys:
                    if k2 != k:
                        if len(node_features[k2]) == 1:
                            res[(k, "related_to", k2)] = [
                                [i, 0] for i in range(prefix_len)
                            ]
                        else:
                            res[(k, "related_to", k2)] = [
                                [i, i] for i in range(prefix_len)
                            ]
            else:
                res[(k, "related_to", k)] = indexes

    return res

In [92]:
from tqdm.notebook import tqdm

In [ ]:

from copy import copy
import torch

from torch_geometric.transforms import ToUndirected
UNDIRECT_TRANSFORMATION = ToUndirected()

""" Builds n-2 HG out of a trace """
def build_prefixes_graph_from_trace(trace, cat_features, real_features):
    X = []  # graphs
    trace = add_new_timestamp(trace) # Normalize time
    node_features = get_node_features(trace, cat_features, real_features)
    
    for prefix in range(1, len(trace)-1):
        
        # For each trace get the prefix slice
        G = HeteroData()
        for k in node_features:
            G[k].x = node_features[k][:(prefix+1)] 

        # Build the edges
        edges_indexes = compute_edges_indexs(node_features, prefix_len=prefix+1)

        # Convert in PyG
        for k in edges_indexes:
            ce = [[], []]
            for i in range(len(edges_indexes[k])):
                ce[0].append(edges_indexes[k][i][0])
                ce[1].append(edges_indexes[k][i][1])
            edges_indexes[k] = ce

        # Assign edges to graph
        for k in edges_indexes:
            G[k].edge_index = tensor(edges_indexes[k], dtype=torch.long)

        # Build labels (aka the targets)
        G.y = {}
        for k in node_features:
            if len(node_features[k]) != 1:  # dynamic
                if k in cat_features:
                    G.y[k] = torch.max(node_features[k][prefix+1], 0)[1].reshape(1,-1)[0].detach().clone()
                else:
                    G.y[k] = node_features[k][prefix+1].reshape(1,-1)[0].detach().clone()
            else:                           # static 
                if k in cat_features:
                    G.y[k] = torch.max(node_features[k][0], 0)[1].reshape(1,-1)[0].detach().clone()
                else:
                    G.y[k] = node_features[k][0].reshape(1,-1)[0].detach().clone()
        
        G = UNDIRECT_TRANSFORMATION(G)
             
        X.append(G)
        
    return X

## Create the graph datasets

In [94]:
case_train_ids = get_case_ids(tab_train)
case_valid_ids = get_case_ids(tab_valid)
case_test_ids = get_case_ids(tab_test)

In [95]:
print(len(case_train_ids))
print(len(case_valid_ids))
print(len(case_test_ids))

4131
1377
1378


In [96]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)

In [97]:
if not os.path.isdir(f"{data_dir_graphs}{dataset}"):
    os.mkdir(f"{data_dir_graphs}{dataset}")

In [98]:
from tqdm.notebook import tqdm

In [ ]:
print("Preparing training dataset...")

X_train = []

for i in tqdm(range(len(case_train_ids))):
    trace = (
        tab_train.query(f"CaseID == '{case_train_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > 2:
        graphs = build_prefixes_graph_from_trace(
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
        )
        
        N_GRAPHS = len(graphs)
        
        for j in range(N_GRAPHS):
            X_train.append(graphs[j])

In [ ]:
torch.save(X_train, f"{GRAPHS_PATH}/train_set.pt")
print("Train Graphs created!\n\n")

In [100]:
del X_train

In [ ]:
print("Preparing validation dataset...")

X_val = []

for i in tqdm(range(len(case_valid_ids))):
    trace = (
        tab_valid.query(f"CaseID == '{case_valid_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > 2:
        graphs = build_prefixes_graph_from_trace(
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
        )
        
        N_GRAPHS = len(graphs)

        
        for j in range(N_GRAPHS):
            X_val.append(graphs[j])

In [ ]:
torch.save(X_val, f"{GRAPHS_PATH}/validation_set.pt")
print("Val Graph created!\n\n")

In [103]:
del X_val

In [ ]:
print("Preparing test dataset...")

X_test = []


for i in tqdm(range(len(case_test_ids))):
    trace = (
        tab_test.query(f"CaseID == '{case_test_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > 2:
        graphs = build_prefixes_graph_from_trace(
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
        )
        
        N_GRAPHS = len(graphs)
        
        for j in range(N_GRAPHS):
            X_test.append(graphs[j])
            

torch.save(X_test, f"{data_dir_graphs}{dataset}/test_set.pt")
print("Test Graphs created!\n\n")

In [ ]:
torch.save(X_test, f"{GRAPHS_PATH}/test_set.pt")
print("Test Graphs created!\n\n")

In [105]:
del X_test